# 02 - Reinforce and Gymnasium

## Block 3: Understanding REINFORCE Intuitively

For this section, we'll use **REINFORCE** (a Policy Gradient algorithm) to solve the *InvertedPendulum* environment. But first, let's understand how REINFORCE works conceptually.

Unlike Q-learning, which builds a "cheat sheet" of values, REINFORCE directly learns a **Policy**—a mathematical rulebook (usually a Neural Network) that tells the agent exactly what to do:

* **The Policy (Actor):** A neural network that looks at the current situation (state) and directly outputs the action to take.
* **Continuous Actions:** Because the force we apply to the pendulum can be any decimal number, we can't just pick from a list of options. Instead, the network outputs the parameters of a probability distribution (like the mean and standard deviation of a bell curve), and the agent *samples* the exact force from that curve.

---

### 1. The Learning Process

Instead of updating after every single step like Q-Learning, REINFORCE learns by reflecting on entire episodes at once:

1.  **Rollout (Experience):** Play an entire episode using the current policy. Keep a log of the states seen, the actions taken, and the rewards received.
2.  **Calculate Returns:** Looking back, calculate the total future reward (discounted return) obtained from each step onwards.
3.  **Update (Gradient Ascent):** Adjust the neural network's weights: *"If this sequence of actions led to a high overall return, tweak the network to make those actions MORE likely next time. If the return was poor, make them LESS likely."*
4.  **Iterate:** Play another episode with the updated, slightly better policy.

> **Why it works:** Policy gradients directly optimize what we care about—the agent's behavior. By bypassing the need to estimate an intermediate Q-value for every single state-action pair, they are incredibly efficient for environments with continuous action spaces.

---

### 2. About the Environment: InvertedPendulum (MuJoCo)


This environment is powered by **MuJoCo**, an advanced physics simulator. It consists of a cart that can be moved left or right along a linear track, with a pole balanced on top of it. The goal is to keep the pendulum standing upright by applying continuous numerical forces to the cart.

*Note: Because the action space is continuous (any force between -3.0 and 3.0), tabular methods like Q-Learning cannot be used here without heavily modifying the environment. This makes it perfect for REINFORCE.*

#### Environment Details

| Feature | Description |
| :--- | :--- |
| **Observation Space** | An array of 4 continuous values: `(cart_position, pole_angle, cart_velocity, pole_angular_velocity)` |
| **Action Space** | A single continuous value between `-3.0` and `3.0` representing the force applied to the cart in Newtons. |
| **Rewards** | `+1` for every timestep the pole remains upright. |
| **Termination** | Episode ends when the pole falls over (the absolute vertical angle exceeds `0.2` radians). |
| **Truncation** | Episode automatically ends after `1000` timesteps. |

📚 *Reference:* [Gymnasium InvertedPendulum Environment Documentation](https://gymnasium.farama.org/environments/mujoco/inverted_pendulum/)

---

### 3. Building a REINFORCE Agent

Let's build our policy gradient agent step by step in the code below. We will need:

* **A Policy Network:** A PyTorch/TensorFlow neural network to map states to action distributions.
* **Action Sampling:** A function to sample an action from the network's output and calculate its log-probability.
* **Return Calculation:** A function to compute the discounted future returns for a full episode trajectory.
* **Network Optimization:** A training step that uses the collected log-probabilities and returns to update the network weights.

### 4. Coding the REINFORCE Agent (Hints)

Now it is your turn to fill in the missing `TODO`s in the PyTorch implementation! Here are the mathematical formulas and PyTorch tricks you will need:

#### Step 1: Sampling and Log-Probabilities
In PyTorch, when you create a distribution like `distrib = Normal(mean, std)`, you can interact with it using built-in methods:
* To get an action, use: `action = distrib.sample()`
* To get the log-probability of that specific action (which we need for the loss function), use: `prob = distrib.log_prob(action)`

#### Step 2: Calculating Discounted Returns
We need to calculate the discounted return ($G_t$) for every step in the episode. We do this backwards, starting from the last reward and moving to the first:
$$G_t = R_t + \gamma \cdot G_{t+1}$$
*Code hint:* Inside the reversed loop, your new `running_g` is the current step's reward plus `self.gamma` multiplied by the previous `running_g`.

#### Step 3: The Policy Gradient Loss Function

The core theorem of REINFORCE states that we update our policy weights using the log-probabilities multiplied by the discounted returns.

Because PyTorch's optimizers are built to *minimize* loss (Gradient Descent), but we want to *maximize* our rewards (Gradient Ascent), we must put a **negative sign** in front of our loss calculation:
$$Loss = -\sum_{t} (\log \pi(A_t | S_t) \cdot G_t)$$
*Code hint:* Multiply your `log_probs` tensor by your `deltas` (returns) tensor, take the sum using `torch.sum()`, and don't forget the minus sign!

In [1]:
!pip install gymnasium[mujoco]

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.1 MB/s eta 0:00:00
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 76.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.6/170.6 kB 15.7 MB/s eta 0:00:00
Using cached importlib_resources-6.5.2-py3-none-any.whl (37 kB)


In [2]:
from __future__ import annotations

import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.distributions.normal import Normal

import gymnasium as gym


plt.rcParams["figure.figsize"] = (10, 5)

In [3]:
class Policy_Network(nn.Module):
    """Parametrized Policy Network."""

    def __init__(self, obs_space_dims: int, action_space_dims: int):
        """Initializes a neural network that estimates the mean and standard deviation
         of a normal distribution from which an action is sampled from.

        Args:
            obs_space_dims: Dimension of the observation space
            action_space_dims: Dimension of the action space
        """
        super().__init__()

        hidden_space1 = 16  # Feel free to change
        hidden_space2 = 32  # Feel free 32, feel free to change

        # Shared Network
        # TODO: Sequential -> 2 x Linear with Tanh is a good starting point
        self.shared_net = nn.Sequential(
            nn.Linear(obs_space_dims, hidden_space1),
            nn.Tanh(),
            nn.Linear(hidden_space1, hidden_space2),
            nn.Tanh(),        )

        # Policy Mean specific: 1x Linear Layer (it gets the input from the shared net and outputs in the dimension of actions)
        self.policy_mean_net = nn.Sequential(
            nn.Linear(hidden_space2, action_space_dims),
        )

        # Policy Std Dev specific: 1x Linear Layer (it gets the input from the shared net and outputs in the dimension of actions)
        self.policy_stddev_net = nn.Sequential(
            nn.Linear(hidden_space2, action_space_dims),
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Conditioned on the observation, returns the mean and standard deviation
         of a normal distribution from which an action is sampled from.

        Args:
            x: Observation from the environment

        Returns:
            action_means: predicted mean of the normal distribution
            action_stddevs: predicted standard deviation of the normal distribution
        """
        shared_features = self.shared_net(x.float())

        action_means = self.policy_mean_net(shared_features)

        action_stddevs = torch.log(
            1 + torch.exp(self.policy_stddev_net(shared_features))
        )

        return action_means, action_stddevs

In [4]:
class REINFORCE:
    """REINFORCE algorithm."""

    def __init__(self, obs_space_dims: int, action_space_dims: int):
        """Initializes an agent that learns a policy via REINFORCE algorithm [1]
        to solve the task at hand (Inverted Pendulum v4).

        Args:
            obs_space_dims: Dimension of the observation space
            action_space_dims: Dimension of the action space
        """

        # Hyperparameters
        self.learning_rate = 1e-4  # Learning rate for policy optimization
        self.gamma = 0.99  # Discount factor
        self.eps = 1e-6  # small number for mathematical stability

        self.probs = []  # Stores probability values of the sampled action
        self.rewards = []  # Stores the corresponding rewards

        self.net = Policy_Network(obs_space_dims, action_space_dims)
        self.optimizer = torch.optim.AdamW(self.net.parameters(), lr=self.learning_rate)

    def sample_action(self, state: np.ndarray) -> float:
        """Returns an action, conditioned on the policy and observation.

        Args:
            state: Observation from the environment

        Returns:
            action: Action to be performed
        """
        state = torch.tensor(np.array([state]))
        action_means, action_stddevs = self.net(state)

        # create a normal distribution from the predicted
        # mean and standard deviation and sample an action
        distrib = Normal(action_means[0] + self.eps, action_stddevs[0] + self.eps)
        # TODO: Sample an action
        action = distrib.sample()
        # TODO: extract log_prob
        prob = distrib.log_prob(action)

        action = action.numpy()

        self.probs.append(prob)

        return action

    def update(self):
        """Updates the policy network's weights."""
        running_g = 0
        gs = []

        # Discounted return (backwards) - Hint: [::-1] will return an array in reverse
        for R in self.rewards[::-1]:
            # TODO: compute running_g following reinforce formula
            running_g = R + self.gamma * running_g
            gs.insert(0, running_g)

        deltas = torch.tensor(gs)

        log_probs = torch.stack(self.probs).squeeze()

        # Update the loss with the mean log probability and deltas
        # TODO: Now, we compute the correct total loss by taking the sum of the element-wise products.
        loss = -torch.sum(log_probs * deltas)

        # Update the policy network
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Empty / zero out all episode-centric/related variables
        self.probs = []
        self.rewards = []

In [5]:
# Run all the below cells to test and show a video of your agent in the environment

In [6]:
# @title
# Create and wrap the environment
env = gym.make("InvertedPendulum-v4")
wrapped_env = gym.wrappers.RecordEpisodeStatistics(env, 50)  # Records episode-reward

total_num_episodes = int(2e3)  # Total number of episodes
# Observation-space of InvertedPendulum-v4 (4)
obs_space_dims = env.observation_space.shape[0]
# Action-space of InvertedPendulum-v4 (1)
action_space_dims = env.action_space.shape[0]
rewards_over_seeds = []

for seed in [1, 2, 3]:  # Fibonacci seeds
    # set seed
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    # Reinitialize agent every seed
    agent = REINFORCE(obs_space_dims, action_space_dims)
    reward_over_episodes = []

    for episode in range(total_num_episodes):
        obs, info = wrapped_env.reset(seed=seed)

        done = False
        while not done:
            #TODO: sample action
            action = agent.sample_action(obs)

            # Step return type - `tuple[ObsType, SupportsFloat, bool, bool, dict[str, Any]]`
            # These represent the next observation, the reward from the step,
            # if the episode is terminated, if the episode is truncated and
            # additional info from the step
            # TODO: step the wrapped_env
            obs, reward, terminated, truncated, info = wrapped_env.step(action)
            agent.rewards.append(reward)

            # End the episode when either truncated or terminated is true
            #  - truncated: The episode duration reaches max number of timesteps
            #  - terminated: Any of the state space values is no longer finite.
            #
            done = terminated or truncated

        reward_over_episodes.append(wrapped_env.return_queue[-1])
        agent.update()

        if episode % 1000 == 0:
            avg_reward = int(np.mean(wrapped_env.return_queue))
            print("Episode:", episode, "Average Reward:", avg_reward)

    rewards_over_seeds.append(reward_over_episodes)

/davinci-1/home/abuonfiglio/MasterPoliTO/Modelli_AI/day1_assingments_RL/.venv/lib/python3.10/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment InvertedPendulum-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Episode: 0 Average Reward: 8
Episode: 1000 Average Reward: 38
Episode: 0 Average Reward: 81
Episode: 1000 Average Reward: 22
Episode: 0 Average Reward: 54
Episode: 1000 Average Reward: 15


In [7]:
# @title
import os
import glob
import io
import base64
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from IPython import display as ipythondisplay

def show_video(directory="./video"):
    """Finds the most recent mp4 video in the directory and displays it."""
    mp4list = glob.glob(f'{directory}/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''<video alt="test" autoplay
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{0}" type="video/mp4" />
                 </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

def test_continuous_agent(agent, env_class, num_episodes=3, video_dir="./video"):
    """Tests the agent and records the episodes to video."""

    # 1. Instantiate the environment with the correct render mode
    if isinstance(env_class, gym.Env):
        base_env = env_class(render_mode="rgb_array")
    else:
        base_env = gym.make(env_class, render_mode="rgb_array")

    # 2. Wrap it to record video.
    # episode_trigger=lambda x: True means we record EVERY episode in this test loop
    env = RecordVideo(base_env, video_folder=video_dir, episode_trigger=lambda x: True)

    total_rewards = []
    successes = 0

    print(f"Starting test for {num_episodes} episodes. Recording to {video_dir}...")

    for ep in range(num_episodes):
        obs, info = env.reset()
        episode_reward = 0
        done = False

        while not done:
            # For continuous agents, get_action should return the deterministic (noise-free) float array
            action = agent.sample_action(obs)

            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated

            if truncated and not terminated:
                successes += 1

        total_rewards.append(episode_reward)
        print(f"Episode {ep + 1}: Reward = {episode_reward:.2f}")

    env.close()

    # Metrics
    win_rate = successes / num_episodes
    average_reward = np.mean(total_rewards)

    print("\n--- Test Results ---")
    print(f"Survival Rate (Avoided Crashing): {win_rate:.1%}")
    print(f"Average Reward: {average_reward:.3f}")
    print(f"Standard Deviation: {np.std(total_rewards):.3f}")

    # Display the video in the notebook
    print("\nDisplaying recorded video:")
    show_video(video_dir)


# To run it:
# test_continuous_agent(agent, "InvertedPendulum-v4", num_episodes=3)

In [9]:
import os
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
test_continuous_agent(agent, "InvertedPendulum-v4", num_episodes=100)

/davinci-1/home/abuonfiglio/MasterPoliTO/Modelli_AI/day1_assingments_RL/.venv/lib/python3.10/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment InvertedPendulum-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(
/davinci-1/home/abuonfiglio/MasterPoliTO/Modelli_AI/day1_assingments_RL/.venv/lib/python3.10/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /davinci-1/home/abuonfiglio/MasterPoliTO/Modelli_AI/day1_assingments_RL/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Starting test for 100 episodes. Recording to ./video...
Episode 1: Reward = 58.00
Episode 2: Reward = 28.00
Episode 3: Reward = 40.00
Episode 4: Reward = 22.00
Episode 5: Reward = 68.00
Episode 6: Reward = 33.00
Episode 7: Reward = 15.00
Episode 8: Reward = 74.00
Episode 9: Reward = 35.00
Episode 10: Reward = 45.00
Episode 11: Reward = 43.00
Episode 12: Reward = 46.00
Episode 13: Reward = 50.00
Episode 14: Reward = 15.00
Episode 15: Reward = 38.00
Episode 16: Reward = 39.00
Episode 17: Reward = 35.00
Episode 18: Reward = 35.00
Episode 19: Reward = 28.00
Episode 20: Reward = 20.00
Episode 21: Reward = 30.00
Episode 22: Reward = 62.00
Episode 23: Reward = 54.00
Episode 24: Reward = 60.00
Episode 25: Reward = 89.00
Episode 26: Reward = 61.00
Episode 27: Reward = 53.00
Episode 28: Reward = 52.00
Episode 29: Reward = 67.00
Episode 30: Reward = 49.00
Episode 31: Reward = 40.00
Episode 32: Reward = 19.00
Episode 33: Reward = 44.00
Episode 34: Reward = 43.00
Episode 35: Reward = 18.00
Episode 